In [ ]:
import json
import re

INPUT_JSON = r"C:\FPTU\doangeo\ocr-pdf-to-text\output\problems\geometry_problems_with_images.json"
OUTPUT_JSON =r"C:\FPTU\doangeo\ocr-pdf-to-text\output\problems\geometry_problems_with_images_tachgiai.json"

CUT_KEYWORDS = [
    "Do ", "Vậy", "Suy ra", "Chứng minh", "Xét ", "Ta có", "Vì "
]

def clean_content(text):
    # tìm (H.x.y)
    match = re.search(r"\(H\.\d+\.\d+\)", text)
    if not match:
        return text.strip()

    h_pos = match.end()
    after = text[h_pos:].strip()

    # nếu sau (H.x.y) còn câu hỏi → giữ
    if any(q in after for q in ["Tính", "Chứng minh", "Hỏi", "Viết", "Tìm"]):
        return text.strip()

    # nếu (H.x.y) nằm giữa câu → giữ
    if not after.startswith("."):
        return text.strip()

    # nếu sau (H.x.y) là lời giải → cắt
    for kw in CUT_KEYWORDS:
        if after.startswith(kw) or f". {kw}" in after:
            return text[:match.start()].strip()

    return text.strip()


with open(INPUT_JSON, "r", encoding="utf-8") as f:
    data = json.load(f)

for item in data:
    item["content"] = clean_content(item["content"])

with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print("✅ Đã làm sạch content và lưu vào:", OUTPUT_JSON)


# V2

In [ ]:
import json
import re

INPUT_JSON = r"C:\FPTU\doangeo\ocr-pdf-to-text\output\problems\geometry_problems_with_images.json"
OUTPUT_JSON =r"C:\FPTU\doangeo\ocr-pdf-to-text\output\problems\geometry_problems_with_images_tachgiai_v2.json"

#  DẤU HIỆU CÓ MÔ TẢ HÌNH HỌC (TRƯỚC H.x.y) 
GEOM_PATTERNS = [
    r"\b[A-Z]{2,}\b",          # AB, AC, ABCD
    r"\b[A-Z]\b",              # A, B, C, D
    r"//|⊥|=",
    r"góc|đường|trung điểm"
]


#  DẤU HIỆU LỜI GIẢI (SAU H.x.y) 
CUT_KEYWORDS = [
    "Do ", "Vậy", "Suy ra", "Ta có", "Vì ",
    "Xét ", "Từ đó", "Giả sử"
]

#  DẤU HIỆU CÂU HỎI → KHÔNG CẮT 
QUESTION_KEYWORDS = ["Tính", "Chứng minh", "Hỏi", "Viết", "Tìm"]

def has_geometry(text):
    """Kiểm tra trước (H.x.y) đã có mô tả hình hay chưa"""
    return any(re.search(p, text) for p in GEOM_PATTERNS)

def is_solution_part(text):
    """Kiểm tra sau (H.x.y) có phải lời giải không"""
    text = text.strip()
    for kw in CUT_KEYWORDS:
        if text.startswith(kw) or f". {kw}" in text:
            return True
    return False

def clean_content(text):
    match = re.search(r"\(H\.\d+\.\d+\)", text)
    if not match:
        return text.strip()

    before = text[:match.start()]
    after = text[match.end():].strip()

    # 1️⃣ Nếu phía sau vẫn còn câu hỏi → giữ
    if any(q in after for q in QUESTION_KEYWORDS):
        return text.strip()

    # 2️⃣ Nếu (H.x.y) nằm giữa câu → giữ
    if not text[match.end():].startswith("."):
        return text.strip()

    # 3️⃣ Nếu trước H.x.y CHƯA có mô tả hình → giữ
    if not has_geometry(before):
        return text.strip()

    # 4️⃣ Nếu sau H.x.y KHÔNG phải lời giải → giữ
    if not is_solution_part(after):
        return text.strip()

    # ✅ ĐỦ ĐIỀU KIỆN → CẮT
    return before.strip()

#  CHẠY PIPELINE 
with open(INPUT_JSON, "r", encoding="utf-8") as f:
    data = json.load(f)

for item in data:
    item["content"] = clean_content(item["content"])

with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print("✅ Hoàn tất! File đã làm sạch:", OUTPUT_JSON)
